In [1]:
# https://www.kaggle.com/competitions/alchemy-aicc-round-4

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from scipy.optimize import linear_sum_assignment

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
train = pd.read_csv("/kaggle/input/alchemy-aicc-round-4/train.csv")
test = pd.read_csv("/kaggle/input/alchemy-aicc-round-4/test.csv")
cands = pd.read_csv("/kaggle/input/alchemy-aicc-round-4/candidates.csv")
cands = cands['result'].values

train.shape, test.shape, cands.shape

((150, 3), (70, 3), (70,))

In [4]:
train.head()

,item1,item2,result
0,bird,fire,phoenix
1,clay,fire,brick
2,fire,water,steam
3,flower,sun,sunflower
4,fire,sand,glass


In [5]:
test.head()

,Id,item1,item2
0,0,solar system,solar system
1,1,alcohol,cactus
2,2,alcohol,water
3,3,alcohol,wheat
4,4,brick,man


In [6]:
extra_train = train.copy()

items1 = extra_train.item1
items2 = extra_train.item2

extra_train.item1, extra_train.item2 = items2, items1

train = pd.concat([train, extra_train], ignore_index=True)

train.tail()

,item1,item2,result
0,bird,fire,phoenix
1,clay,fire,brick
2,fire,water,steam
3,flower,sun,sunflower
4,fire,sand,glass


In [7]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_checkpoint = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=1).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
text = "bird + fire = phoenix."
inputs = tokenizer(text, return_tensors='pt')
inputs = {k: v.to(device) for k, v in inputs.items()}
output = model(**inputs)
output

SequenceClassifierOutput(loss=None, logits=tensor([[-0.0490]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [9]:
all_words = set()

for i, row in train.iterrows():
    all_words.add(row['item1'])
    all_words.add(row['item2'])
    all_words.add(row['result'])
    
for i, row in test.iterrows():
    all_words.add(row['item1'])
    all_words.add(row['item2'])

for cand in cands:
    all_words.add(cand)

all_words = sorted(list(all_words))
n_words = len(all_words)
n_words

318

In [10]:
idx2word, word2idx = dict(), dict()

for i, word in enumerate(all_words):
    idx2word[i] = word
    word2idx[word] = i

In [11]:
positive_samples = np.empty((train.shape[0], 3), dtype='int64')

for i, row in train.iterrows():
    positive_samples[i] = [
        word2idx[row[c]] for c in train.columns
    ]

def is_positive(sample):    
    if isinstance(sample, list):
        sample = np.array(sample)
    return bool(np.any(np.all(sample == positive_samples, axis=1)))

positive_samples.shape

(300, 3)

In [12]:
def generate_batch(batch_size=32):
    assert batch_size % 2 == 0, 'Batch size should be even'
    assert batch_size//2 <= positive_samples.shape[0], 'Batch size can\'t be bigger than number of positive samples'

    pos_idx = np.random.choice(positive_samples.shape[0], size=batch_size//2, replace=False)
    positives = positive_samples[pos_idx]
    negatives = np.random.randint(n_words, size=(batch_size//2, 3))
    for i in range(negatives.shape[0]):
        while is_positive(negatives[i]):
            negatives[i] = np.random.randint(n_words, size=(3))

    X = np.concatenate([positives, negatives], axis=0)
    y = np.concatenate([np.ones(batch_size//2), np.zeros(batch_size//2)])

    indices = np.arange(batch_size)
    np.random.shuffle(indices)

    X, y = X[indices], y[indices]

    texts = []
    for i in range(batch_size):
        texts.append(f"{idx2word[X[i, 0]]} + {idx2word[X[i, 1]]} = {idx2word[X[i, 2]]}.")
        
    inputs = tokenizer(texts, padding=True, return_tensors='pt')
    return inputs, torch.tensor(y).unsqueeze(1)

In [13]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

In [14]:
def train(model, optimizer, criterion, n_batches=100, epochs=10, log_rate=1, batch_size=32):
    history = {
        'train_loss': [],
        'train_acc': []
    }
    
    model.train()
    for epoch in tqdm(range(epochs), desc='Epoch'):
        i, rloss, all_y, all_preds = 0, 0, [], []
        for batch_id in (pbar := tqdm(range(n_batches), desc='Batch', leave=False)):
            inputs, y = generate_batch(batch_size)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            y = y.to(device)
            logits = model(**inputs).logits
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).to(torch.int64)
            loss = criterion(logits, y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            rloss += loss.item()
            i += 1
            all_y.extend(y.tolist())
            all_preds.extend(preds.tolist())
            acc = accuracy_score(all_y, all_preds)
            pbar.set_postfix({'loss': f'{rloss/i:.5f}', 'acc': f'{acc:.5f}'})

        history['train_loss'].append(rloss/i)
        history['train_acc'].append(acc)

        if (epoch+1) % log_rate == 0:
            print(f'Epoch: {epoch+1}/{epochs} | Loss: {rloss/i:.5f} | Acc: {acc:.5f}')

    return history

In [15]:
history = train(model, optimizer, criterion, epochs=10, log_rate=1)

Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Batch:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch: 1/10 | Loss: 0.27452 | Acc: 0.90250


Batch:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch: 2/10 | Loss: 0.07733 | Acc: 0.97594


...

Epoch: 8/10 | Loss: 0.02204 | Acc: 0.99344


Batch:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch: 9/10 | Loss: 0.02227 | Acc: 0.99469


Batch:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch: 10/10 | Loss: 0.01836 | Acc: 0.99625


In [16]:
scores = np.zeros((test.shape[0], cands.shape[0]))

model.eval()

for i, row in tqdm(test.iterrows(), total=len(test)):
    texts = [f"{row['item1']} + {row['item2']} = {c}." for c in cands]
    inputs = tokenizer(texts, padding=True, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits)
        scores[i] = probs.squeeze().cpu().numpy()

scores.shape

  0%|          | 0/70 [00:00<?, ?it/s]

(70, 70)

In [17]:
row_ind, col_ind = linear_sum_assignment(scores, maximize=True)
row_ind, col_ind

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
        51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
        68, 69]),
 array([45, 40, 55, 68, 65, 16, 42, 37,  3, 27, 54,  6, 58,  8, 48,  9,  1,
        41, 52, 22, 36, 25, 32, 33, 17, 35, 62, 23, 10, 47, 44, 69, 38, 11,
        67,  5, 28, 12, 15, 13, 24, 18, 30,  7, 59, 60, 49, 21,  4, 46, 26,
        50, 20,  2, 39, 66, 14, 64, 43, 63, 53, 51,  0, 61, 31, 34, 57, 29,
        19, 56]))

In [18]:
predictions = [cands[c] for c in col_ind]

submission = pd.DataFrame({
    'Id': test['Id'],
    'result': predictions
})
submission.to_csv('submission.csv', index=False)
print(f'Submission shape: {submission.shape}')
print(submission.head())

Submission shape: (70, 2)
   Id     result
0   0  satellite
1   1      radar
2   2    tequila
3   3      wheat
4   4      vodka
